# 06 — Comparación válida y entrega de resultados

## Decisión que responde la demo

El trabajo de explotación de datos produjo una lista de campaña más útil. Esta comparación sólo vale porque los modelos usan el mismo algoritmo, objetivo y observaciones de prueba.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style='whitegrid', palette='deep')
def find_root():
    for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/retail_customer_intelligence')]:
        if (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError('No se encontró la raíz del proyecto. Abre la carpeta retail_customer_intelligence, no sólo el archivo .ipynb.')
ROOT = find_root()
sys.path.insert(0, str(ROOT / 'src'))
from retail_demo import *
ensure_folders(ROOT)
print('Proyecto:', ROOT)

## Comprobar comparabilidad antes de comparar una métrica

La clave de una observación es CustomerID y snapshot_date: el mismo cliente puede existir en varios cortes. Si ambos modelos no evalúan exactamente las mismas observaciones, ninguna diferencia de métrica es interpretable.

In [ ]:
original_result = pd.read_parquet(ROOT / "artifacts/predicciones_original.parquet")
processed_result = pd.read_parquet(ROOT / "artifacts/predicciones_procesado.parquet")
leaky_result = pd.read_parquet(ROOT / "artifacts/predicciones_fuga_didactica.parquet")
with open(ROOT / "reports/metricas_modelo_original.json", encoding="utf-8") as file:
    m_original = json.load(file)
with open(ROOT / "reports/metricas_modelo_procesado.json", encoding="utf-8") as file:
    m_processed = json.load(file)
with open(ROOT / "reports/metricas_modelo_fuga_didactica.json", encoding="utf-8") as file:
    m_leaky = json.load(file)

key = ["CustomerID", "snapshot_date", "compra_futura", "gasto_futuro"]
assert set(map(tuple, original_result[key].to_numpy())) == set(map(tuple, processed_result[key].to_numpy()))
assert set(map(tuple, original_result[key].to_numpy())) == set(map(tuple, leaky_result[key].to_numpy()))
joined_valid = original_result.merge(processed_result, on=key, suffixes=("_original", "_procesado"))
dummy = no_purchase_baseline(joined_valid.compra_futura)
print(f"Comparación válida: {len(joined_valid):,} observaciones cliente-corte idénticas.")

## Scorecard: separar validez de una cifra llamativa

La regla nula puede parecer buena por accuracy cuando la mayoría no compra. La fuga puede parecer extraordinaria porque cuenta el futuro. Sólo los dos modelos temporalmente válidos responden la pregunta sobre el efecto de procesar los datos.

In [ ]:
scorecard = pd.DataFrame({
    "regla nula: nadie compra": dummy,
    "baseline: datos originales": m_original["clasificacion"],
    "modelo: datos procesados": m_processed["clasificacion"],
    "experimento con fuga — NO USAR": m_leaky["clasificacion"],
}).T.drop(columns="matriz_confusion")
scorecard.insert(0, "validez", ["válido, pero inútil", "válido", "válido", "inválido"])
display(scorecard.round(3))

valid_comparison = scorecard.loc[
    ["baseline: datos originales", "modelo: datos procesados"]
].drop(columns="validez")
delta = valid_comparison.loc["modelo: datos procesados"] - valid_comparison.loc["baseline: datos originales"]
display(delta.rename("cambio procesado - original").to_frame().round(3))

## Evaluar la lista completa

La matriz de confusión depende de un umbral. Una curva precision–recall y la calidad del ranking muestran si los compradores se concentran en la parte alta de la lista, que es lo que realmente se usaría en una campaña.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, metric, title in zip(
    axes[:2],
    [m_original["clasificacion"], m_processed["clasificacion"]],
    ["Baseline: datos originales", "Modelo: datos procesados"],
):
    sns.heatmap(np.array(metric["matriz_confusion"]), annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set(title=title, xlabel="predicción", ylabel="real")
PrecisionRecallDisplay.from_predictions(
    joined_valid.compra_futura, joined_valid.probabilidad_compra_original, name="originales", ax=axes[2]
)
PrecisionRecallDisplay.from_predictions(
    joined_valid.compra_futura, joined_valid.probabilidad_compra_procesado, name="procesados", ax=axes[2]
)
axes[2].set_title("Precision–recall en septiembre")
plt.tight_layout()

metrics_to_plot = ["balanced_accuracy", "precision", "recall", "f1", "pr_auc"]
plot_data = valid_comparison[metrics_to_plot].T.reset_index(names="métrica").melt(
    id_vars="métrica", var_name="dataset", value_name="valor"
)
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=plot_data, x="métrica", y="valor", hue="dataset", ax=ax)
ax.set(ylim=(0, 1), title="Sólo modelos temporalmente válidos")
plt.tight_layout()

## Resultado operativo: calidad del top 10 por ciento

En una campaña se contacta una lista limitada por presupuesto. Medimos precisión, lift, compradores capturados y gasto futuro observado en el top 10 por ciento. Estas cifras miden la capacidad de priorizar; no estiman el efecto causal de enviar una promoción.

In [ ]:
ranking_summary = pd.DataFrame({
    "datos originales": m_original["ranking_por_probabilidad"],
    "datos procesados": m_processed["ranking_por_probabilidad"],
}).T
display(ranking_summary.round(3))

ranking_delta = ranking_summary.loc["datos procesados", [
    "precision_at_k", "lift_at_k", "compras_capturadas", "porcentaje_gasto_futuro_capturado"
]] - ranking_summary.loc["datos originales", [
    "precision_at_k", "lift_at_k", "compras_capturadas", "porcentaje_gasto_futuro_capturado"
]]
display(ranking_delta.rename("cambio procesado - original").to_frame().round(3))

fig, ax = plt.subplots(figsize=(9, 4))
ranking_summary[["precision_at_k", "lift_at_k", "porcentaje_gasto_futuro_capturado"]].T.plot.bar(
    ax=ax, rot=0, color=["#bdbdbd", "#2a9d8f"]
)
ax.set(title="Calidad de la lista que contactaría al 10 por ciento", ylabel="proporción o factor de lift")
plt.tight_layout()

## Entregar una lista que se pueda revisar

La salida se ordena por probabilidad de compra y mantiene campos de contexto. En una aplicación real se añadirían consentimiento, frecuencia máxima, costos y una prueba A/B: el ranking predice propensión, no demuestra que el mensaje provocará una compra.

In [ ]:
profiles = pd.read_parquet(ROOT / "data/processed/modelo_procesado.parquet")
profile_columns = [
    "CustomerID", "snapshot_date", "recencia_dias", "frecuencia_facturas",
    "gasto_historico", "gasto_30d", "tendencia_gasto_30d", "tasa_devolucion",
]
campaign = processed_result.merge(
    profiles.loc[profiles.particion.eq("test"), profile_columns],
    on=["CustomerID", "snapshot_date"], how="left",
).sort_values("probabilidad_compra", ascending=False).reset_index(drop=True)
campaign["orden_campaña"] = np.arange(1, len(campaign) + 1)
top_count = m_processed["ranking_por_probabilidad"]["clientes_contactados"]
campaign["segmento"] = np.where(
    campaign["orden_campaña"].le(top_count),
    "prioridad alta: top 10 por ciento",
    "no contactar en primera ola",
)
campaign["explicación"] = np.where(
    campaign.recencia_dias.le(campaign.recencia_dias.median()),
    "actividad reciente e historial disponible",
    "historial disponible; menor actividad reciente",
)
columns = [
    "orden_campaña", "CustomerID", "probabilidad_compra", "segmento",
    "recencia_dias", "frecuencia_facturas", "gasto_historico", "gasto_30d",
    "tendencia_gasto_30d", "tasa_devolucion", "explicación",
]
delivery = campaign[columns]
delivery.to_csv(ROOT / "reports/campana_clientes.csv", index=False)
display(delivery.head(15).round(3))
print("Archivo entregable:", ROOT / "reports/campana_clientes.csv")

## Cierre de la demostración

La mejora no provino de un modelo más complejo. Provino de entender la unidad de análisis, separar ventas de cancelaciones y devoluciones, construir perfiles antes del corte, validar en el futuro y evaluar una lista priorizada.

El experimento con fuga es intencional: enseña que una métrica extraordinaria puede ser evidencia de un pipeline mal construido. El paso siguiente en una aplicación real es un experimento A/B sobre la lista priorizada.

Entregables: reports/campana_clientes.csv, métricas JSON, predicciones, modelos serializados y bitácoras del pipeline.